### Lab 3.1: Basic Neural Network in PyTorch - Solution

Let's create a linear classifier one more time, but using PyTorch's automatic differentiation and optimization algorithms.  Then you will extend the perceptron into a multi-layer perceptron (MLP).

In [1]:
import numpy as np
import torch

We need to explicitly tell PyTorch when creating a tensor that we are interested in later computing its gradient

In [2]:
a = torch.tensor(5.,requires_grad=True)
a

tensor(5., requires_grad=True)

In [3]:
b = torch.tensor(6.,requires_grad=True)
c = 2*a+3*b
c

tensor(28., grad_fn=<AddBackward0>)

To extract the gradients, we first need to call `backward()`.

In [4]:
c.backward()

Now to get the gradient of any variable with respect to `c`, we simply access the `grad` attribute of that variable.

In [5]:
a.grad

tensor(2.)

In [6]:
b.grad

tensor(3.)

Let's load and format the Palmer penguins dataset for multi-class classification.

In [7]:
from palmerpenguins import load_penguins
from matplotlib import pyplot as plt

In [8]:
df = load_penguins()

# drop rows with missing values
df.dropna(inplace=True)

# get two features
X = df[['flipper_length_mm','bill_length_mm']].values

# convert species labels to integers
y = df['species'].map({'Adelie':0,'Chinstrap':1,'Gentoo':2}).values

To make the learning algorithm work more smoothly, we we will subtract the mean of each feature.

Here `np.mean` calculates a mean, and `axis=0` tells NumPy to calculate the mean over the rows (calculate the mean of each column).

In [9]:
X -= np.mean(X,axis=0)

Now we will convert our `X` and `y` arrays to torch Tensors.

In [10]:
X = torch.tensor(X).float()
y = torch.tensor(y).long()

In [11]:
from torch import nn

The `torch.nn.Sequential` class creates a feed-forward network from a list of `nn.Module` objects.  Here we provide a single `nn.Linear` class which performs an affine transformation ($Wx+b$) so that we will have a linear classifier.

In [12]:
linear_model = torch.nn.Sequential(
    torch.nn.Linear(2,3), # two inputs, three outputs
)

Now we create a cross-entropy loss function object and a stochastic gradient descent (SGD) optimizer.

In [13]:
loss_fn = torch.nn.CrossEntropyLoss()

In [15]:
lr = 1e-2
opt = torch.optim.SGD(linear_model.parameters(), lr=lr)

Finally we can iteratively optimize the model.

In [16]:
epochs = 100
for epoch in range(epochs):
    opt.zero_grad() # zero out the gradients

    z = linear_model(X) # compute z values
    loss = loss_fn(z,y) # compute loss

    loss.backward() # compute gradients

    opt.step() # apply gradients
    
    

    print(f'epoch {epoch}: loss is {loss.item()}')

epoch 0: loss is 7.984398365020752
epoch 1: loss is 6.735891819000244
epoch 2: loss is 5.732630252838135
epoch 3: loss is 4.794509410858154
epoch 4: loss is 3.8864009380340576
epoch 5: loss is 3.0004665851593018
epoch 6: loss is 2.14597749710083
epoch 7: loss is 1.3930896520614624
epoch 8: loss is 0.9791645407676697
epoch 9: loss is 0.8467085361480713
epoch 10: loss is 0.7721283435821533
epoch 11: loss is 0.7163155674934387
epoch 12: loss is 0.6700190901756287
epoch 13: loss is 0.6298154592514038
epoch 14: loss is 0.5941104292869568
epoch 15: loss is 0.5620403289794922
epoch 16: loss is 0.5330743193626404
epoch 17: loss is 0.5068480968475342
epoch 18: loss is 0.483083039522171
epoch 19: loss is 0.4615480303764343
epoch 20: loss is 0.44204002618789673
epoch 21: loss is 0.42437469959259033
epoch 22: loss is 0.40838271379470825
epoch 23: loss is 0.39390650391578674
epoch 24: loss is 0.38079988956451416
epoch 25: loss is 0.36892616748809814
epoch 26: loss is 0.358157217502594
epoch 27: los

### Exercises

Extend the above code to implement an MLP with a single hidden layer of size 100.


In [24]:
class MLP(nn.Module):
    def __init__(self, hidden_size=100):
        super(MLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(2, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 3)
        )
        
    def forward(self, x):
        return self.network(x)

In [39]:
mlp_model = MLP()
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.SGD(mlp_model.parameters(), lr=lr)

epochs = 100
for epoch in range(epochs):
    opt.zero_grad() # zero out the gradients

    z = mlp_model(X) # compute z values
    loss = loss_fn(z,y) # compute loss

    loss.backward() # compute gradients

    opt.step() # apply gradients

    print(f'epoch {epoch}: loss is {loss.item()}')

epoch 0: loss is 1.2322840690612793
epoch 1: loss is 0.5995346307754517
epoch 2: loss is 0.38334357738494873
epoch 3: loss is 0.3037398159503937
epoch 4: loss is 0.27549102902412415
epoch 5: loss is 0.2550451457500458
epoch 6: loss is 0.23974333703517914
epoch 7: loss is 0.2279811054468155
epoch 8: loss is 0.21869757771492004
epoch 9: loss is 0.21116548776626587
epoch 10: loss is 0.20488658547401428
epoch 11: loss is 0.19952307641506195
epoch 12: loss is 0.19484995305538177
epoch 13: loss is 0.19071592390537262
epoch 14: loss is 0.18701671063899994
epoch 15: loss is 0.18367774784564972
epoch 16: loss is 0.18064363300800323
epoch 17: loss is 0.17787137627601624
epoch 18: loss is 0.1753263920545578
epoch 19: loss is 0.17298096418380737
epoch 20: loss is 0.17081186175346375
epoch 21: loss is 0.1687997281551361
epoch 22: loss is 0.16692811250686646
epoch 23: loss is 0.16518285870552063
epoch 24: loss is 0.16355182230472565
epoch 25: loss is 0.16202448308467865
epoch 26: loss is 0.160591304

Write code to compute the accuracy of each model.

Can you get the MLP to outperform the linear model?

In [45]:
def get_accuracy(model, X, y):
    model.eval()

    total = 0
    correct = 0

    with torch.no_grad():
        z = model(X)
        predictions = z.argmax(dim=1)
        correct += (y == predictions).sum()
        total += y.size(0)

    return float(correct / total)

print(f"Linear model accuracy: {100 * get_accuracy(linear_model, X, y):.2f}%")
print(f"MLP model accuracy: {100 * get_accuracy(mlp_model, X, y):.2f}%")

Linear model accuracy: 93.99%
MLP model accuracy: 95.20


The MLP model was able to outperform the linear model in accuracy by 1.21